In [22]:
import numpy as np
import pandas as pd
import time
import contextlib
import io
from pathlib import Path

import matplotlib.pyplot as plt

from raiselab.core import dvqe

import gurobipy as gp
from gurobipy import GRB


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path("dvqe_n30_easy_diagonal_results")
OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

N = 15
QUBO_SEED = 30001
DVQE_SEED = 130001

# Easy diagonal QUBO:
#     min z^T Q z + q^T z
# Since Q is diagonal and z_i^2 = z_i, each variable is independent.
# The coefficient of z_i is Q_ii + q_i.
# If Q_ii + q_i < 0, the optimal value is z_i = 1.
# Otherwise, the optimal value is z_i = 0.

DIAGONAL_SCALE = 1.0
LINEAR_SCALE = 1.0

# Strong but not too expensive D-BH settings for an easy n=30 case.
DBH_SETTINGS = dict(
    depth=2,
    lr=0.5,
    max_iters=100,
    rel_tol=1e-4,
    num_shots=768,
    final_shots=8000,
    warm_start_population=16,
    warm_start_iters=25,
    warm_start_shots=512,
    energy_mode="cvar",
    cvar_alpha=0.1,
    verbose=False,
)

DBH_METHOD = dict(
    method="D-BH",
    mode="distributed",
    init_type=2,
    qpu_qubit_config=[4, 4, 4, 4, 4, 4, 4, 4],  # total capacity = 32, enough for n = 30
)

GUROBI_TIME_LIMIT = None
GUROBI_THREADS = 1
GUROBI_OUTPUT = 0

EXACT_SUCCESS_TOL = 1e-8
NEAR_SUCCESS_TOL = 1e-4

ROTATION_PARAM_DECIMALS = 3


# ============================================================
# QUBO HELPERS
# ============================================================

def qubo_energy(Q, q, z):
    """
    Compute QUBO energy:
        C(z) = z^T Q z + q^T z
    """
    z = np.asarray(z, dtype=float)
    return float(z.T @ Q @ z + q.T @ z)


def scale_qubo(Q, q):
    """
    Scale QUBO coefficients for DVQE numerical stability.
    This does not change the minimizer because the objective is divided
    by a positive constant.
    """
    scale = max(
        float(np.max(np.abs(Q))) if Q.size > 0 else 0.0,
        float(np.max(np.abs(q))) if q.size > 0 else 0.0,
        1.0,
    )

    return Q / scale, q / scale, scale


def hamming_distance(z1, z2):
    z1 = np.asarray(z1, dtype=int)
    z2 = np.asarray(z2, dtype=int)
    return int(np.sum(z1 != z2))


def bitstring(z):
    return "".join(map(str, np.asarray(z, dtype=int).tolist()))


def generate_easy_diagonal_qubo(n, seed, diagonal_scale=1.0, linear_scale=1.0):
    """
    Generate an easy diagonal QUBO.

    Objective:
        C(z) = z^T Q z + q^T z

    Since Q is diagonal and z_i^2 = z_i for binary variables,
        C(z) = sum_i (Q_ii + q_i) z_i.

    Therefore, the problem separates into n independent binary decisions.
    """
    rng = np.random.default_rng(seed)

    Q = np.zeros((n, n), dtype=float)

    # Positive diagonal values.
    diag = rng.uniform(0.1, diagonal_scale, size=n)
    np.fill_diagonal(Q, diag)

    # Mixed-sign linear terms.
    # Some variables prefer z_i = 1, others prefer z_i = 0.
    q = rng.uniform(-linear_scale, linear_scale, size=n)

    # Avoid nearly zero combined coefficients to reduce ties.
    coeff = diag + q
    for i in range(n):
        if abs(coeff[i]) < 0.15:
            q[i] += 0.25 if coeff[i] >= 0 else -0.25

    Q = 0.5 * (Q + Q.T)

    return Q, q


def solve_diagonal_qubo_exact(Q, q):
    """
    Direct exact solution for a diagonal QUBO.
    This is included only as a sanity check.
    """
    diag = np.diag(Q)
    coeff = diag + q

    z = (coeff < 0.0).astype(int)
    obj = qubo_energy(Q, q, z)

    return z, obj, coeff


# ============================================================
# GUROBI MIQP SOLVER
# ============================================================

def solve_qubo_gurobi_miqp(Q, q, time_limit=None, threads=1, output_flag=0):
    """
    Solve the diagonal QUBO with Gurobi MIQP.

    Problem:
        min z^T Q z + q^T z
        s.t. z_i in {0,1}
    """
    n = q.shape[0]

    model = gp.Model("n30_easy_diagonal_qubo_miqp")
    model.Params.OutputFlag = output_flag
    model.Params.Threads = threads
    model.Params.NonConvex = 2
    model.Params.MIPGap = 0.0

    if time_limit is not None:
        model.Params.TimeLimit = time_limit

    z = model.addVars(n, vtype=GRB.BINARY, name="z")

    obj = gp.QuadExpr()

    # Because Q is diagonal, only diagonal terms are needed.
    for i in range(n):
        if abs(Q[i, i]) > 0.0:
            obj += float(Q[i, i]) * z[i] * z[i]

    for i in range(n):
        if abs(q[i]) > 0.0:
            obj += float(q[i]) * z[i]

    model.setObjective(obj, GRB.MINIMIZE)

    start = time.time()
    model.optimize()
    runtime = time.time() - start

    status_code = model.Status
    status_name = {
        GRB.OPTIMAL: "optimal",
        GRB.TIME_LIMIT: "time_limit",
        GRB.INFEASIBLE: "infeasible",
        GRB.INF_OR_UNBD: "inf_or_unbd",
        GRB.UNBOUNDED: "unbounded",
    }.get(status_code, f"status_{status_code}")

    if model.SolCount == 0:
        return {
            "z_best": None,
            "objective": np.nan,
            "runtime_sec": runtime,
            "status": status_name,
            "mip_gap": np.nan,
            "node_count": np.nan,
        }

    z_best = np.array([int(round(z[i].X)) for i in range(n)], dtype=int)
    objective = qubo_energy(Q, q, z_best)

    try:
        mip_gap = float(model.MIPGap)
    except Exception:
        mip_gap = np.nan

    try:
        node_count = float(model.NodeCount)
    except Exception:
        node_count = np.nan

    return {
        "z_best": z_best,
        "objective": objective,
        "runtime_sec": runtime,
        "status": status_name,
        "mip_gap": mip_gap,
        "node_count": node_count,
    }


# ============================================================
# D-BH DVQE SOLVER
# ============================================================

def solve_qubo_dbh_n30(Q, q):
    """
    Solve the n=30 diagonal QUBO using distributed DVQE with Black Hole warm start.
    """
    Q_train, q_train, qubo_scale = scale_qubo(Q, q)

    start = time.time()

    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        z_best, final_circuit, hist = dvqe(
            mode=DBH_METHOD["mode"],
            Q=Q_train,
            q_linear=q_train,
            init_type=DBH_METHOD["init_type"],
            qpu_qubit_config=DBH_METHOD["qpu_qubit_config"],
            seed=DVQE_SEED,
            **DBH_SETTINGS,
        )

    runtime = time.time() - start

    z_best = np.asarray(z_best, dtype=int)
    objective = qubo_energy(Q, q, z_best)

    return {
        "z_best": z_best,
        "objective": objective,
        "runtime_sec": runtime,
        "hist": hist,
        "hist_size": len(hist) if hist is not None else np.nan,
        "qubo_scale": qubo_scale,
        "final_circuit": final_circuit,
        "status": "ok",
    }


# ============================================================
# CIRCUIT DRAWING HELPERS
# ============================================================

def rounded_circuit_for_drawing(circuit, decimals=3):
    """
    Return a copy of the circuit where numeric gate parameters are rounded
    for cleaner publication-quality circuit drawings.

    This only changes the displayed circuit copy. It does not change the
    trained circuit used for optimization or objective evaluation.
    """
    qc_round = circuit.copy()

    for inst in qc_round.data:
        operation = inst.operation

        if not hasattr(operation, "params"):
            continue

        new_params = []

        for p in operation.params:
            try:
                val = float(p)
                new_params.append(round(val, decimals))
            except Exception:
                new_params.append(p)

        operation.params = new_params

    return qc_round


# ============================================================
# FIGURE SAVING
# ============================================================

def save_trained_circuit_figures(final_circuit):
    """
    Save high-quality figures of the trained distributed ansatz.

    The displayed copy of the circuit rounds all numeric rotation parameters
    to ROTATION_PARAM_DECIMALS digits after the decimal point for publication
    readability.

    The text export uses UTF-8 on Windows because Qiskit text drawings contain
    Unicode box-drawing characters.
    """
    try:
        from qiskit.visualization import circuit_drawer
    except Exception as e:
        print("Could not import qiskit.visualization.circuit_drawer.")
        print(str(e))
        return

    circuit_to_draw = rounded_circuit_for_drawing(
        final_circuit,
        decimals=ROTATION_PARAM_DECIMALS,
    )

    # ------------------------------------------------------------
    # Full folded circuit figure
    # ------------------------------------------------------------
    try:
        fig_full = circuit_drawer(
            circuit_to_draw,
            output="mpl",
            fold=60,
            scale=0.70,
            style={
                "fontsize": 10,
                "subfontsize": 8,
                "displaytext": {
                    "rx": "Rx",
                    "ry": "Ry",
                    "rz": "Rz",
                    "cx": "CX",
                    "cz": "CZ",
                },
            },
        )

        fig_full.set_size_inches(18, 14)
        fig_full.savefig(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal_full.png",
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.05,
        )
        fig_full.savefig(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal_full.pdf",
            bbox_inches="tight",
            pad_inches=0.05,
        )
        plt.close(fig_full)

    except Exception as e:
        print("Could not save full MPL circuit figure.")
        print(str(e))

    # ------------------------------------------------------------
    # Compact manuscript figure
    # ------------------------------------------------------------
    try:
        fig_compact = circuit_drawer(
            circuit_to_draw,
            output="mpl",
            fold=35,
            scale=0.85,
            style={
                "fontsize": 12,
                "subfontsize": 9,
                "displaytext": {
                    "rx": "Rx",
                    "ry": "Ry",
                    "rz": "Rz",
                    "cx": "CX",
                    "cz": "CZ",
                },
            },
        )

        fig_compact.set_size_inches(16, 18)
        fig_compact.savefig(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal_compact.png",
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.05,
        )
        fig_compact.savefig(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal_compact.pdf",
            bbox_inches="tight",
            pad_inches=0.05,
        )
        plt.close(fig_compact)

    except Exception as e:
        print("Could not save compact MPL circuit figure.")
        print(str(e))

    # ------------------------------------------------------------
    # Text export with UTF-8 and rounded parameters
    # ------------------------------------------------------------
    try:
        with open(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal_text.txt",
            "w",
            encoding="utf-8",
        ) as f:
            f.write(str(circuit_to_draw.draw(output="text", fold=140)))
    except Exception as e:
        print("Could not save text circuit drawing.")
        print(str(e))

    # ------------------------------------------------------------
    # QASM export with original unrounded trained circuit
    # ------------------------------------------------------------
    try:
        with open(
            OUTPUT_DIR / "trained_distributed_ansatz_n30_easy_diagonal.qasm",
            "w",
            encoding="utf-8",
        ) as f:
            f.write(final_circuit.qasm())
    except Exception as e:
        print("Could not save QASM file.")
        print(str(e))

    print("Saved trained circuit outputs in:")
    print(OUTPUT_DIR)


def save_top_histogram_plot(hist, Q, q, top_k=20):
    """
    Save a plot of the best sampled bitstrings.
    """
    if hist is None or len(hist) == 0:
        return

    rows = []

    for bitstr_raw, count in hist.items():
        bitstr_clean = str(bitstr_raw).replace(" ", "")

        # Some circuit outputs may include extra classical bits.
        # Keep the rightmost N bits if needed.
        if len(bitstr_clean) > N:
            bitstr_clean = bitstr_clean[-N:]

        z = np.array([int(c) for c in bitstr_clean], dtype=int)

        if len(z) != N:
            continue

        cost = qubo_energy(Q, q, z)
        rows.append((bitstr_clean, count, cost))

    if len(rows) == 0:
        return

    df = pd.DataFrame(rows, columns=["bitstring", "count", "cost"])
    df = df.sort_values("cost", ascending=True).head(top_k).copy()

    df.to_csv(
        OUTPUT_DIR / "top_sampled_bitstrings_n30_easy_diagonal.csv",
        index=False,
    )

    labels = [s[:8] + r"$\cdots$" + s[-4:] for s in df["bitstring"]]
    costs = df["cost"].to_numpy()

    plt.figure(figsize=(12, 5))
    plt.bar(range(len(costs)), costs)
    plt.xticks(range(len(costs)), labels, rotation=60, ha="right", fontsize=9)
    plt.ylabel("QUBO objective value", fontsize=12)
    plt.xlabel("Sampled bitstrings", fontsize=12)
    plt.title("Best sampled bitstrings from trained D-BH circuit", fontsize=13)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "top_sampled_bitstrings_n30_easy_diagonal.png",
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    plt.savefig(
        OUTPUT_DIR / "top_sampled_bitstrings_n30_easy_diagonal.pdf",
        bbox_inches="tight",
        pad_inches=0.05,
    )
    plt.close()


# ============================================================
# MAIN RUNNER
# ============================================================

def run_n30_easy_diagonal_example():
    print("=" * 100)
    print("N=30 EASY DIAGONAL QUBO EXAMPLE WITH GUROBI AND D-BH")
    print("=" * 100)
    print(f"n = {N}")
    print(f"QUBO seed = {QUBO_SEED}")
    print(f"D-BH QPU config = {DBH_METHOD['qpu_qubit_config']}")
    print(f"D-BH settings = {DBH_SETTINGS}")
    print(f"Rotation parameters shown with {ROTATION_PARAM_DECIMALS} decimals in figures")
    print("=" * 100)

    Q, q = generate_easy_diagonal_qubo(
        n=N,
        seed=QUBO_SEED,
        diagonal_scale=DIAGONAL_SCALE,
        linear_scale=LINEAR_SCALE,
    )

    np.save(OUTPUT_DIR / "Q_n30_easy_diagonal.npy", Q)
    np.save(OUTPUT_DIR / "q_n30_easy_diagonal.npy", q)

    # ------------------------------------------------------------
    # Direct diagonal solution
    # ------------------------------------------------------------
    z_direct, f_direct, coeff = solve_diagonal_qubo_exact(Q, q)

    print("\nDirect diagonal solution:")
    print(f"  obj={f_direct:.10f}")
    print(f"  z={bitstring(z_direct)}")

    # ------------------------------------------------------------
    # Gurobi solve
    # ------------------------------------------------------------
    print("\nSolving with Gurobi MIQP...")
    gurobi_result = solve_qubo_gurobi_miqp(
        Q=Q,
        q=q,
        time_limit=GUROBI_TIME_LIMIT,
        threads=GUROBI_THREADS,
        output_flag=GUROBI_OUTPUT,
    )

    print(
        f"Gurobi MIQP | status={gurobi_result['status']} | "
        f"obj={gurobi_result['objective']:.10f} | "
        f"time={gurobi_result['runtime_sec']:.4f}s | "
        f"gap={gurobi_result['mip_gap']}"
    )

    # ------------------------------------------------------------
    # D-BH solve
    # ------------------------------------------------------------
    print("\nSolving with D-BH DVQE...")
    dbh_result = solve_qubo_dbh_n30(Q, q)

    print(
        f"D-BH DVQE | obj={dbh_result['objective']:.10f} | "
        f"time={dbh_result['runtime_sec']:.4f}s | "
        f"hist_size={dbh_result['hist_size']}"
    )

    # ------------------------------------------------------------
    # Comparison
    # ------------------------------------------------------------
    z_gurobi = gurobi_result["z_best"]
    z_dbh = dbh_result["z_best"]

    raw_gap = float(dbh_result["objective"] - gurobi_result["objective"])
    abs_gap = max(0.0, raw_gap)
    rel_gap = abs_gap / max(1.0, abs(float(gurobi_result["objective"])))

    exact_success = bool(abs_gap <= EXACT_SUCCESS_TOL)
    near_success = bool(abs_gap <= NEAR_SUCCESS_TOL)

    ham_to_gurobi = hamming_distance(z_dbh, z_gurobi)
    ham_to_direct = hamming_distance(z_dbh, z_direct)

    same_gurobi = bool(bitstring(z_dbh) == bitstring(z_gurobi))
    same_direct = bool(bitstring(z_dbh) == bitstring(z_direct))

    print("\nComparison:")
    print(f"  raw gap D-BH - Gurobi = {raw_gap:.6e}")
    print(f"  absolute gap          = {abs_gap:.6e}")
    print(f"  relative gap          = {rel_gap:.6e}")
    print(f"  exact success         = {exact_success}")
    print(f"  near success          = {near_success}")
    print(f"  Hamming to Gurobi     = {ham_to_gurobi}")
    print(f"  Hamming to direct     = {ham_to_direct}")
    print(f"  same as Gurobi        = {same_gurobi}")
    print(f"  same as direct        = {same_direct}")

    # ------------------------------------------------------------
    # Save scalar and solution results
    # ------------------------------------------------------------
    result_row = {
        "n": N,
        "qubo_seed": QUBO_SEED,
        "dvqe_seed": DVQE_SEED,

        "direct_objective": f_direct,
        "direct_z": bitstring(z_direct),

        "gurobi_objective": gurobi_result["objective"],
        "gurobi_runtime_sec": gurobi_result["runtime_sec"],
        "gurobi_status": gurobi_result["status"],
        "gurobi_mip_gap": gurobi_result["mip_gap"],
        "gurobi_node_count": gurobi_result["node_count"],
        "gurobi_z": bitstring(z_gurobi),

        "dbh_objective": dbh_result["objective"],
        "dbh_runtime_sec": dbh_result["runtime_sec"],
        "dbh_hist_size": dbh_result["hist_size"],
        "dbh_qubo_scale": dbh_result["qubo_scale"],
        "dbh_z": bitstring(z_dbh),

        "raw_gap_dbh_minus_gurobi": raw_gap,
        "abs_gap": abs_gap,
        "rel_gap": rel_gap,
        "exact_success": exact_success,
        "near_success": near_success,
        "hamming_to_gurobi": ham_to_gurobi,
        "hamming_to_direct": ham_to_direct,
        "same_solution_gurobi": same_gurobi,
        "same_solution_direct": same_direct,

        "depth": DBH_SETTINGS["depth"],
        "lr": DBH_SETTINGS["lr"],
        "max_iters": DBH_SETTINGS["max_iters"],
        "num_shots": DBH_SETTINGS["num_shots"],
        "final_shots": DBH_SETTINGS["final_shots"],
        "warm_start_population": DBH_SETTINGS["warm_start_population"],
        "warm_start_iters": DBH_SETTINGS["warm_start_iters"],
        "warm_start_shots": DBH_SETTINGS["warm_start_shots"],
        "energy_mode": DBH_SETTINGS["energy_mode"],
        "cvar_alpha": DBH_SETTINGS["cvar_alpha"],
        "qpu_qubit_config": str(DBH_METHOD["qpu_qubit_config"]),
        "rotation_param_decimals_in_figures": ROTATION_PARAM_DECIMALS,
    }

    pd.DataFrame([result_row]).to_csv(
        OUTPUT_DIR / "n30_easy_diagonal_dbh_vs_gurobi_result.csv",
        index=False,
    )

    with open(
        OUTPUT_DIR / "n30_easy_diagonal_solution_strings.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write("Direct diagonal solution:\n")
        f.write(result_row["direct_z"] + "\n\n")
        f.write("Gurobi solution:\n")
        f.write(result_row["gurobi_z"] + "\n\n")
        f.write("D-BH solution:\n")
        f.write(result_row["dbh_z"] + "\n\n")
        f.write(f"Exact success: {exact_success}\n")
        f.write(f"Near success: {near_success}\n")
        f.write(f"Abs gap: {abs_gap:.12e}\n")
        f.write(f"Rel gap: {rel_gap:.12e}\n")
        f.write(f"Hamming to Gurobi: {ham_to_gurobi}\n")
        f.write(f"Hamming to direct: {ham_to_direct}\n")

    # ------------------------------------------------------------
    # Save coefficient table
    # ------------------------------------------------------------
    coeff_df = pd.DataFrame({
        "i": np.arange(N),
        "Q_ii": np.diag(Q),
        "q_i": q,
        "combined_coeff_Qii_plus_qi": coeff,
        "direct_z_i": z_direct,
        "gurobi_z_i": z_gurobi,
        "dbh_z_i": z_dbh,
    })

    coeff_df.to_csv(
        OUTPUT_DIR / "n30_easy_diagonal_coefficients.csv",
        index=False,
    )

    # ------------------------------------------------------------
    # Save figures
    # ------------------------------------------------------------
    print("\nSaving trained distributed ansatz figures...")
    save_trained_circuit_figures(dbh_result["final_circuit"])

    print("\nSaving sampled-bitstring plot...")
    save_top_histogram_plot(dbh_result["hist"], Q, q, top_k=20)

    print("\nSaved all outputs in:")
    print(OUTPUT_DIR.resolve())

    return result_row, gurobi_result, dbh_result




In [23]:
result_row, gurobi_result, dbh_result = run_n30_easy_diagonal_example()

N=30 EASY DIAGONAL QUBO EXAMPLE WITH GUROBI AND D-BH
n = 15
QUBO seed = 30001
D-BH QPU config = [4, 4, 4, 4, 4, 4, 4, 4]
D-BH settings = {'depth': 2, 'lr': 0.5, 'max_iters': 100, 'rel_tol': 0.0001, 'num_shots': 768, 'final_shots': 8000, 'warm_start_population': 16, 'warm_start_iters': 25, 'warm_start_shots': 512, 'energy_mode': 'cvar', 'cvar_alpha': 0.1, 'verbose': False}
Rotation parameters shown with 3 decimals in figures

Direct diagonal solution:
  obj=-2.2301858846
  z=001001011100000

Solving with Gurobi MIQP...
Gurobi MIQP | status=optimal | obj=-2.2301858846 | time=0.0000s | gap=0.0

Solving with D-BH DVQE...
D-BH DVQE | obj=-2.2301858846 | time=7240.2314s | hist_size=3131

Comparison:
  raw gap D-BH - Gurobi = 0.000000e+00
  absolute gap          = 0.000000e+00
  relative gap          = 0.000000e+00
  exact success         = True
  near success          = True
  Hamming to Gurobi     = 0
  Hamming to direct     = 0
  same as Gurobi        = True
  same as direct        = True
